In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os

# ------------------------------
# Configuration
# ------------------------------


DATA_DIR = r"E:\yr3\year_3_ai_ml\year_3_ai_ml\dstse\src\core\data"

# Data repository directory
DATA_REPO = os.path.join(DATA_DIR, "data_repo")
FEATURE_STORE=os.path.join(DATA_DIR, "feature_store")

# Input weather_data file
INPUT_FILE = [
    os.path.join(FEATURE_STORE, "1_weather_data_with_categories.csv"),
    os.path.join(DATA_REPO, "anonymized_transactions_2023.csv")
    ]

FILES_OUT=[
    {
        "ensemble":{
                "random_forest": {
            "regression": os.path.join(FEATURE_STORE, "rf_reg_categorised_weather_data_with_features.csv"),
            "classification": os.path.join(FEATURE_STORE, "rf_clf_categorised_weather_data_with_features.csv")
        },
                 "xgboost": os.path.join(FEATURE_STORE, "xgb_categorised_weather_data_with_features.csv")
        },  
        "data_types":{
            "time_series": os.path.join(FEATURE_STORE, "ts_categorised_weather_data_with_features.csv")
        },
        "problem_types":{
                "regression": os.path.join(FEATURE_STORE, "reg_categorised_weather_data_with_features.csv"),
                "classification": os.path.join(FEATURE_STORE, "clf_categorised_weather_data_with_features.csv")
        },
        "other":[
            os.path.join(FEATURE_STORE, "rd_level_data_with_eng_features.csv"),
            os.path.join(FEATURE_STORE, "dock_level_data_with_eng_features.csv"),
            os.path.join(FEATURE_STORE, "traffic_for_lecture_system.csv")
        ]
    }
]



### WEATHER DATA FEATURE ENGINEERING

In [5]:
# Load weather data
print(f"Reading input file: {INPUT_FILE[0]}")
df = pd.read_csv(INPUT_FILE[0], encoding='utf-8', parse_dates=['datetime'])

Reading input file: E:\yr3\year_3_ai_ml\year_3_ai_ml\dstse\src\core\data\feature_store\1_weather_data_with_categories.csv


In [6]:
df.describe()

,Total Rainfall,Dry bulb,Wet bulb,Humidity,Wind speed,datetime,log_rainfall,hour,rain_category
count,1098.000000,1098.000000,1098.000000,1098.000000,1098.000000,1098,1098.000000,1098.000000,1098.000000
mean,6.736066,23.745537,20.141166,73.387978,2.183060,2023-05-01 01:34:25.573770,1.077702,12.000000,1.213115
min,0.000000,17.000000,16.800000,44.000000,0.000000,2023-03-01 08:00:00,0.000000,8.000000,0.000000
25%,0.000000,21.400000,19.500000,62.000000,0.000000,2023-03-31 12:15:00,0.000000,10.000000,0.000000
50%,0.450000,24.300000,20.400000,70.000000,2.000000,2023-05-01 00:00:00,0.370969,12.000000,1.000000
75%,7.300000,26.000000,21.000000,84.750000,4.000000,2023-05-31 11:45:00,2.116256,14.000000,2.000000
max,83.300000,30.200000,22.400000,100.000000,8.000000,2023-06-30 16:00:00,4.434382,16.000000,3.000000
std,12.799316,2.851436,0.946509,13.421188,2.003235,NaN,1.293912,2.583165,1.168615


In [7]:
df.dtypes

Date                         str
Time                         str
Total Rainfall           float64
Dry bulb                 float64
Wet bulb                 float64
Humidity                 float64
Wind speed                 int64
datetime          datetime64[us]
log_rainfall             float64
hour                       int64
rain_category              int64
rain_label                   str
dtype: object

#### WEATHER DATA TIME SERIES FEATURE ENGINEERING


In [8]:
ts_df = df.copy()

In [9]:
# Basic time features
ts_df['hour'] = ts_df['datetime'].dt.hour
ts_df['date'] = ts_df['datetime'].dt.date
ts_df['date']= pd.to_datetime(ts_df['date'])  # Ensure 'date' is datetime type
ts_df['month'] = ts_df['datetime'].dt.month
ts_df['day'] = ts_df['datetime'].dt.day
ts_df['year'] = ts_df['datetime'].dt.year
ts_df['dayofweek'] = ts_df['datetime'].dt.dayofweek
ts_df['dayofyear'] = ts_df['datetime'].dt.dayofyear

# Cyclical encoding 
ts_df['hour_sin'] = np.sin(2 * np.pi * (ts_df['hour'] - 8) / 9)  # shift to start at 8
ts_df['hour_cos'] = np.cos(2 * np.pi * (ts_df['hour'] - 8) / 9)
ts_df['month_sin'] = np.sin(2 * np.pi * (ts_df['month'] - 1) / 12)
ts_df['month_cos'] = np.cos(2 * np.pi * (ts_df['month'] - 1) / 12)

# Lag features 
# Create lagged versions of key variables from previous hours
for lag in [1, 2, 3]:
    ts_df[f'rainfall_lag_{lag}'] = ts_df['Total Rainfall'].shift(lag)
    ts_df[f'humidity_lag_{lag}'] = ts_df['Humidity'].shift(lag)
    ts_df[f'drybulb_lag_{lag}'] = ts_df['Dry bulb'].shift(lag)

# Rolling statistics (average over last 3 hours)
ts_df['rainfall_roll_3h'] = ts_df['Total Rainfall'].rolling(window=3, min_periods=1).mean()
ts_df['humidity_roll_3h'] = ts_df['Humidity'].rolling(window=3, min_periods=1).mean()
ts_df['drybulb_roll_3h'] = ts_df['Dry bulb'].rolling(window=3, min_periods=1).mean()

In [10]:
# Check for missing values after feature engineering 
# since lag and rolling features can introduce NaNs at the start of the dataset.
# these NaN can be handled later (e.g., by imputation or dropping) depending on the modeling approach.
# but we drop them for now in the third notebook from this one to keep things simple.
ts_df.isnull().sum()

Date                0
Time                0
Total Rainfall      0
Dry bulb            0
Wet bulb            0
Humidity            0
Wind speed          0
datetime            0
log_rainfall        0
hour                0
rain_category       0
rain_label          0
date                0
month               0
day                 0
year                0
dayofweek           0
dayofyear           0
hour_sin            0
hour_cos            0
month_sin           0
month_cos           0
rainfall_lag_1      1
humidity_lag_1      1
drybulb_lag_1       1
rainfall_lag_2      2
humidity_lag_2      2
drybulb_lag_2       2
rainfall_lag_3      3
humidity_lag_3      3
drybulb_lag_3       3
rainfall_roll_3h    0
humidity_roll_3h    0
drybulb_roll_3h     0
dtype: int64

In [11]:
# Check rows with NaN values (should be the first few due to lag features)
ts_df[ts_df.isnull().any(axis=1)]

,Date,Time,Total Rainfall,Dry bulb,Wet bulb,Humidity,Wind speed,datetime,log_rainfall,hour,...,drybulb_lag_1,rainfall_lag_2,humidity_lag_2,drybulb_lag_2,rainfall_lag_3,humidity_lag_3,drybulb_lag_3,rainfall_roll_3h,humidity_roll_3h,drybulb_roll_3h
0,3/1/2023,8:00,0.0,18.5,18.0,96.0,0,2023-03-01 08:00:00,0.0,8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,96.0,18.50
1,3/1/2023,9:00,0.0,18.6,18.0,94.0,2,2023-03-01 09:00:00,0.0,9,...,18.5,NaN,NaN,NaN,NaN,NaN,NaN,0.0,95.0,18.55
2,3/1/2023,10:00,0.0,21.4,19.4,83.0,2,2023-03-01 10:00:00,0.0,10,...,18.6,0.0,96.0,18.5,NaN,NaN,NaN,0.0,91.0,19.50


In [12]:
# Drop rows with NaN from lag features
ts_df = ts_df.dropna().reset_index(drop=True)

ts_df[ts_df.isnull().any(axis=1)]

,Date,Time,Total Rainfall,Dry bulb,Wet bulb,Humidity,Wind speed,datetime,log_rainfall,hour,...,drybulb_lag_1,rainfall_lag_2,humidity_lag_2,drybulb_lag_2,rainfall_lag_3,humidity_lag_3,drybulb_lag_3,rainfall_roll_3h,humidity_roll_3h,drybulb_roll_3h


In [13]:
ts_df.to_csv(FILES_OUT[0]['data_types']['time_series'], index=False)
print("Saved data with categories to 'categorised_weather_data_with_features.csv'")

Saved data with categories to 'categorised_weather_data_with_features.csv'


#### WEATHER DATA REGRESSION TARGET FEATURE ENGINEERING

In [14]:
reg_df=pd.DataFrame()
reg_df['datetime'] = df['datetime']

In [15]:
# Target for regression: log(1 + Total Rainfall)
# added a small constant (1) to handle zero rainfall cases and avoid log(0) cases under no rain
reg_df['log_rainfall'] = np.log1p(df['Total Rainfall'])

In [16]:
reg_df.isnull().sum()

datetime        0
log_rainfall    0
dtype: int64

In [17]:
# Save the dataframe with the new category column
print("shape of reg_df:", reg_df.shape)
print("X columns:", reg_df.columns.tolist())

reg_df.to_csv(FILES_OUT[0]['problem_types']['regression'], index=False)
print("Saved data with categories to 'reg_categorised_weather_data_with_features.csv'")

shape of reg_df: (1098, 2)
X columns: ['datetime', 'log_rainfall']
Saved data with categories to 'reg_categorised_weather_data_with_features.csv'


# TRAFFIC DATA

In [18]:
# df has columns: station, start (datetime), date, etc.

df = pd.read_csv(INPUT_FILE[1], encoding='utf-8', parse_dates=['start','date'])

C:\Users\KIYIMBA FAHAD\AppData\Local\Temp\ipykernel_13328\2147039653.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(INPUT_FILE[1], encoding='utf-8', parse_dates=['start','date'])


In [19]:
df.describe()

,date,amount,excess_time,start,old_transaction_id,release_station,part,cash,off_system,digital_time_before,timestamp_closed,request_by
count,9957,9957.000000,9209.000000,9957,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0
mean,2023-10-15 15:40:45.917445,475.896354,-1.814421,2026-05-01 13:35:43.730039,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
min,2023-08-17 00:00:00,0.000000,-174316.000000,2026-05-01 07:21:00,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
25%,2023-09-16 00:00:00,500.000000,2.000000,2026-05-01 11:39:00,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
50%,2023-10-14 00:00:00,500.000000,13.000000,2026-05-01 13:46:00,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
75%,2023-11-11 00:00:00,500.000000,18.000000,2026-05-01 15:51:00,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
max,2023-12-20 00:00:00,15000.000000,108295.000000,2026-05-01 18:28:00,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
std,NaN,417.598879,2139.903180,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN


In [20]:
df.dtypes

agent_name                        str
agent                             str
date                   datetime64[us]
agent_code                        str
phone_number                      str
type                              str
residence                         str
payment_method                    str
email                             str
timestamp                         str
amount                        float64
name                              str
bike_number                       str
station                           str
excess_time                   float64
start                  datetime64[us]
coarse_code                       str
duration                          str
return_agent                      str
return_timestamp                  str
staff_student                     str
end                               str
destination                       str
return_duration                   str
gender                            str
old_transaction_id            float64
release_stat

In [21]:
tfc001_df = df.copy()

In [22]:
# check that 'start' is a datetime with the correct date to prevent assigning of current date
tfc001_df['start'] = pd.to_datetime(
    tfc001_df['date'].astype(str) + ' ' + 
    tfc001_df['start'].dt.strftime('%H:%M:%S')
)

# Round start to nearest 15 min
tfc001_df['time_rounded'] = tfc001_df['start'].dt.round('15min')
# Group to get raw counts
raw = tfc001_df.groupby(['station', 'date', 'time_rounded']).size().reset_index(name='count')

In [23]:
print("Raw data sample (first 10 rows):")
print(raw.head(10))
print("\nUnique stations in raw:", raw['station'].unique())
print("\nRaw records for station 'Africa' on 2023-08-17:")
print(raw[(raw['station'] == 'Africa') & (raw['date'] == '2023-08-17')])

Raw data sample (first 10 rows):
  station       date        time_rounded  count
0  Africa 2023-08-17 2023-08-17 13:00:00      1
1  Africa 2023-08-17 2023-08-17 13:15:00      1
2  Africa 2023-08-22 2023-08-22 15:15:00      1
3  Africa 2023-08-24 2023-08-24 12:15:00      3
4  Africa 2023-08-24 2023-08-24 13:30:00      1
5  Africa 2023-08-24 2023-08-24 15:15:00      2
6  Africa 2023-08-25 2023-08-25 10:00:00      1
7  Africa 2023-08-25 2023-08-25 10:30:00      1
8  Africa 2023-08-25 2023-08-25 12:00:00      1
9  Africa 2023-08-25 2023-08-25 13:15:00      1

Unique stations in raw: <StringArray>
[         'Africa',           'CEDAT',           'CONAS',         'Complex',
    'Eastern Gate',         'Library',     'Livingstone',       'Main Gate',
     'Mary Stuart',        'Mitchell',   'Swimming Pool', 'University Hall']
Length: 12, dtype: str

Raw records for station 'Africa' on 2023-08-17:
  station       date        time_rounded  count
0  Africa 2023-08-17 2023-08-17 13:00:00      1
1

In [24]:
# For each station, create complete time grid
stations = raw['station'].unique()
all_dates = raw['date'].unique()
all_times = pd.date_range('00:00', '23:45', freq='15min').time

expanded = []
for station in stations:
    for date in all_dates:
        for t in all_times:
            expanded.append({'station': station, 'date': date, 'time': t})
expanded_df = pd.DataFrame(expanded)
expanded_df['datetime'] = pd.to_datetime(expanded_df['date'].astype(str) + ' ' + expanded_df['time'].astype(str))



In [25]:
print("Sample raw time_rounded:", raw['time_rounded'].iloc[0])
print("Sample expanded datetime:", expanded_df['datetime'].iloc[0])



Sample raw time_rounded: 2023-08-17 13:00:00
Sample expanded datetime: 2023-08-17 00:00:00


In [26]:
# Merge with raw counts
merged = expanded_df.merge(raw, left_on=['station', 'datetime'], right_on=['station', 'time_rounded'], how='left')
merged['count'] = merged['count'].fillna(np.nan)

# Interpolate per station
merged = merged.sort_values(['station', 'datetime'])
merged['count'] = merged.groupby('station')['count'].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
).round().fillna(0).astype(int)

# Fill any remaining NaNs at the boundaries with 0
merged['count'] = merged['count'].fillna(0).astype(int)



In [27]:
# Extract temporal features from datetime

# Basic time components
def extract_time_features(df):
    df['hour'] = df['datetime'].dt.hour
    df['minute'] = df['datetime'].dt.minute
    df['date'] = df['datetime'].dt.date
    df['day_of_month'] = df['datetime'].dt.day
    df['month'] = df['datetime'].dt.month
    df['year'] = df['datetime'].dt.year

    # Day of week (Monday=0, Sunday=6)
    df['day_of_week'] = df['datetime'].dt.dayofweek

    # Day name for readability
    df['day_name'] = df['datetime'].dt.day_name()

    # Is weekend? (Saturday=5, Sunday=6)
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    return df



In [28]:
# View the results
def view_extracted_features(df):
    print("Sample of extracted features:")
    print(df[['datetime', 'hour', 'day_of_week', 'day_name', 'is_weekend', 'time_of_day', 'is_rush_hour']].head(10))

    # Check distributions
    print("\n=== Distribution by hour ===")
    print(df['hour'].value_counts().sort_index())

    print("\n=== Distribution by day of week ===")
    print(df['day_name'].value_counts())

    print("\n=== Weekend vs Weekday ===")
    print(df['is_weekend'].value_counts())

    print("\n=== Time of day distribution ===")
    print(df['time_of_day'].value_counts())


In [29]:
# Time of day category
def time_of_day(hour):
    if 5 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 17:
        return 'afternoon'
    elif 17 <= hour < 21:
        return 'evening'
    else:
        return 'night'

In [30]:
# removed these five classes ecause they were causing imbalance issues in the classification model 
# and we had very few samples in the 'Low' and 'Medium' categories,
# which made it difficult for the model to learn meaningful patterns from those classes. 
# By simplifying to just 'Low', 'Normal', and 'Very high', 
# we can focus on the more distinct congestion levels and improve model performance.

# determine congestion

# def count_to_congestion(count,low_threshold, medium_threshold,normal_threshold,high_threshold,very_high_threshold):
#     """
#     Convert motorcycle/vehice count to congestion level based on observed data range 2-7.
#     """
    
#     if count <= low_threshold:
#         return 'Low'
#     elif count <= medium_threshold:
#         return 'Medium'
#     elif count <= normal_threshold:
#         return 'Normal'
#     elif count <= high_threshold:
#         return 'High'
#     else:
#         return 'Very high'


def count_to_congestion(count,low_threshold, normal_threshold):
    """
    Convert motorcycle/vehice count to congestion level based on observed data range 2-7.
    """
    
    if count <= low_threshold:
        return 'Low'
    
    elif count <= normal_threshold:
        return 'Normal'
    
    else:
        return 'Very high'

In [31]:
merged=extract_time_features(merged)

In [32]:
merged['time_of_day'] = merged['hour'].apply(time_of_day)

# Rush hour indicator
merged['is_rush_hour'] = merged['hour'].apply(
    lambda x: 1 if (7 <= x <= 9) or (16 <= x <= 19) else 0
).astype(int)

# Business hours (8am - 6pm)
merged['is_business_hours'] = merged['hour'].apply(
    lambda x: 1 if 8 <= x <= 18 else 0
).astype(int)


In [33]:
view_extracted_features(merged)

Sample of extracted features:
             datetime  hour  day_of_week  day_name  is_weekend time_of_day  \
0 2023-08-17 00:00:00     0            3  Thursday           0       night   
1 2023-08-17 00:15:00     0            3  Thursday           0       night   
2 2023-08-17 00:30:00     0            3  Thursday           0       night   
3 2023-08-17 00:45:00     0            3  Thursday           0       night   
4 2023-08-17 01:00:00     1            3  Thursday           0       night   
5 2023-08-17 01:15:00     1            3  Thursday           0       night   
6 2023-08-17 01:30:00     1            3  Thursday           0       night   
7 2023-08-17 01:45:00     1            3  Thursday           0       night   
8 2023-08-17 02:00:00     2            3  Thursday           0       night   
9 2023-08-17 02:15:00     2            3  Thursday           0       night   

   is_rush_hour  
0             0  
1             0  
2             0  
3             0  
4             0  
5  

In [34]:
# Apply calibration based on time
def get_calibration_factor(row):
    hour = row['hour']
    day = row['day_of_week']
    
    # Weekend vs weekday
    if day >= 5:  # Saturday, Sunday
        base_factor = 6
    else:  # Weekday
        base_factor = 8
    
    # Rush hour adjustment
    if (7 <= hour <= 9) or (16 <= hour <= 19):
        return base_factor + 2  # More cars during rush hour
    elif 22 <= hour or hour <= 5:  # Late night
        return base_factor - 3  # Mostly motorcycles at night
    else:
        return base_factor

merged['calibration_factor'] = merged.apply(get_calibration_factor, axis=1)
merged['estimated_vehicle_count'] = merged['count'] * merged['calibration_factor']

# removed these five classes ecause they were causing imbalance issues in the classification model 
# and we had very few samples in the 'Low' and 'Medium' categories,
# which made it difficult for the model to learn meaningful patterns from those classes. 
# By simplifying to just 'Low', 'Normal', and 'Very high', 
# we can focus on the more distinct congestion levels and improve model performance.

# so we only calculate the 20th and 60th percentiles to create three categories: Low, Normal, and Very high.

# Compute percentiles from actual counts
# # Calculate percentiles (20th, 40th, 60th, 80th)
# low_threshold = merged['estimated_vehicle_count'].quantile(0.20)      # 20th percentile
# medium_threshold = merged['estimated_vehicle_count'].quantile(0.40)   # 40th percentile
# normal_threshold = merged['estimated_vehicle_count'].quantile(0.80)   # 80th percentile
# high_threshold = merged['estimated_vehicle_count'].quantile(0.90)     # 90th percentile
# very_high_threshold = merged['estimated_vehicle_count'].quantile(0.95)   # 95th percentile

# Calculate percentiles (20th, 80th)
low_threshold = merged['estimated_vehicle_count'].quantile(0.20)      # 20th percentilemedium_threshold = merged['estimated_vehicle_count'].quantile(0.40)   # 40th percentile
normal_threshold = merged['estimated_vehicle_count'].quantile(0.60)   # 60th percentile

# def wrapper(x):
#     return count_to_congestion(x, low_threshold, medium_threshold, normal_threshold, high_threshold, very_high_threshold)

def wrapper(x):
    return count_to_congestion(x, low_threshold, normal_threshold)

merged['congestion'] = merged['estimated_vehicle_count'].apply(wrapper)

print(f"Thresholds based on data distribution:")
print(f"Low/Medium boundary (20th percentile): {low_threshold:.2f}")
print(f"Normal/High boundary (60th percentile): {normal_threshold:.2f}")


Thresholds based on data distribution:
Low/Medium boundary (20th percentile): 5.00
Normal/High boundary (60th percentile): 8.00


In [35]:
# Save the dataframe with the new
merged.to_csv(FILES_OUT[0]['other'][0], index=False)
print("Saved data with categories to 'rd_level_data_with_eng_features.csv'")

Saved data with categories to 'rd_level_data_with_eng_features.csv'


In [36]:
# create a mapping of stations to roads based on the provided station names and their corresponding roads
road_mapping = {
    'Africa': 'Bombo Road',
    'CEDAT': 'University Road',
    'CONAS': 'University Road',
    'Complex': 'Sir Apolo Road',
    'Eastern Gate': 'Northern Bypass',
    'Library': 'University Road',
    'Livingstone': 'Sir Apolo Road',
    'Main Gate': 'Bombo Road',
    'Mary Stuart': 'Northern Bypass',
    'Mitchell': 'Sir Apolo Road',
    'Swimming Pool': 'University Road',
    'University Hall': 'Bombo Road'
}

merged['road'] = merged['station'].map(road_mapping)

In [37]:
road_traffic = merged.groupby(['road', 'datetime']).agg({
    'estimated_vehicle_count': 'sum'
}).reset_index()

road_traffic.dtypes

road                                  str
datetime                   datetime64[us]
estimated_vehicle_count             int64
dtype: object

In [38]:
# we did the same here, elimating the five classes and only calculating the 20th and 60th percentiles to create three categories: Low, Normal, and Very high. 
 
# compute the dock level congestion thresholds based on the distribution of estimated vehicle counts

# # Compute percentiles from actual counts
# low_threshold = road_traffic['estimated_vehicle_count'].quantile(0.20)      # 20th percentile
# medium_threshold = road_traffic['estimated_vehicle_count'].quantile(0.40)   # 40th percentile
# normal_threshold = road_traffic['estimated_vehicle_count'].quantile(0.60)   # 60th percentile
# high_threshold = road_traffic['estimated_vehicle_count'].quantile(0.80)     # 80th percentile
# very_high_threshold = road_traffic['estimated_vehicle_count'].quantile(0.95)   # 90th percentile

# Compute percentiles from actual counts
low_threshold = road_traffic['estimated_vehicle_count'].quantile(0.20)      # 20th percentile
normal_threshold = road_traffic['estimated_vehicle_count'].quantile(0.60)   # 60th percentile


def wrapper(x):
    return count_to_congestion(x, low_threshold, normal_threshold)

road_traffic['congestion'] = road_traffic['estimated_vehicle_count'].apply(wrapper)

print(f"Thresholds based on data distribution:")
print(f"Low/Medium boundary (20th percentile): {low_threshold:.2f}")
print(f"Normal/High boundary (60th percentile): {normal_threshold:.2f}")

Thresholds based on data distribution:
Low/Medium boundary (20th percentile): 15.00
Normal/High boundary (60th percentile): 30.00


In [39]:
# extract time features from datetime

extract_time_features(road_traffic)

,road,datetime,estimated_vehicle_count,congestion,hour,minute,date,day_of_month,month,year,day_of_week,day_name,is_weekend
0,Bombo Road,2023-08-17 00:00:00,15,Low,0,0,2023-08-17,17,8,2023,3,Thursday,0
1,Bombo Road,2023-08-17 00:15:00,15,Low,0,15,2023-08-17,17,8,2023,3,Thursday,0
2,Bombo Road,2023-08-17 00:30:00,15,Low,0,30,2023-08-17,17,8,2023,3,Thursday,0
3,Bombo Road,2023-08-17 00:45:00,15,Low,0,45,2023-08-17,17,8,2023,3,Thursday,0
4,Bombo Road,2023-08-17 01:00:00,15,Low,1,0,2023-08-17,17,8,2023,3,Thursday,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
40315,University Road,2023-12-20 22:45:00,20,Normal,22,45,2023-12-20,20,12,2023,2,Wednesday,0
40316,University Road,2023-12-20 23:00:00,20,Normal,23,0,2023-12-20,20,12,2023,2,Wednesday,0
40317,University Road,2023-12-20 23:15:00,20,Normal,23,15,2023-12-20,20,12,2023,2,Wednesday,0
40318,University Road,2023-12-20 23:30:00,20,Normal,23,30,2023-12-20,20,12,2023,2,Wednesday,0


In [40]:
road_traffic['time_of_day'] = road_traffic['hour'].apply(time_of_day)

# Rush hour indicator
road_traffic['is_rush_hour'] = road_traffic['hour'].apply(
    lambda x: 1 if (7 <= x <= 9) or (16 <= x <= 19) else 0
).astype(int)

# Business hours (8am - 6pm)
road_traffic['is_business_hours'] = road_traffic['hour'].apply(
    lambda x: 1 if 8 <= x <= 18 else 0
).astype(int)


In [41]:
view_extracted_features(road_traffic)

Sample of extracted features:
             datetime  hour  day_of_week  day_name  is_weekend time_of_day  \
0 2023-08-17 00:00:00     0            3  Thursday           0       night   
1 2023-08-17 00:15:00     0            3  Thursday           0       night   
2 2023-08-17 00:30:00     0            3  Thursday           0       night   
3 2023-08-17 00:45:00     0            3  Thursday           0       night   
4 2023-08-17 01:00:00     1            3  Thursday           0       night   
5 2023-08-17 01:15:00     1            3  Thursday           0       night   
6 2023-08-17 01:30:00     1            3  Thursday           0       night   
7 2023-08-17 01:45:00     1            3  Thursday           0       night   
8 2023-08-17 02:00:00     2            3  Thursday           0       night   
9 2023-08-17 02:15:00     2            3  Thursday           0       night   

   is_rush_hour  
0             0  
1             0  
2             0  
3             0  
4             0  
5  

In [42]:
# Save the dataframe with the new
road_traffic.to_csv(FILES_OUT[0]['other'][1], index=False)
print("Saved data with categories to 'dock_level_data_with_eng_features.csv'")

Saved data with categories to 'dock_level_data_with_eng_features.csv'


In [43]:
# Create production traffic data
# for bayesian reasoning
traffic_for_lectures = road_traffic[[
    'road', 
    'datetime',
    'month',
    'hour',
    'day_of_week',
    'is_weekend',
    'is_rush_hour',
    'estimated_vehicle_count',
    'congestion'
]].copy()

# Sort for consistency
traffic_for_lectures = traffic_for_lectures.sort_values(['road', 'datetime'])

# Save
traffic_for_lectures.to_csv(FILES_OUT[0]['other'][2], index=False)



In [44]:
road_traffic.to_csv(FILES_OUT[0]['other'][1], index=False)
print("Saved data with categories to 'traffic_for_lecture_system.csv'")

Saved data with categories to 'traffic_for_lecture_system.csv'
